# 🧊 Image → 3D Model
**Made by Claude** · Runs on Colab free tier (T4 GPU)

### Two modes:
- **`subject`** — isolates the main object and reconstructs it as a full 3D model (InstantMesh)
- **`scene`** — converts the entire photo into a 3D terrain with the photo as texture (MiDaS)

⚠️ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running!

In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────
MODE = "scene"   # "subject" or "scene"
# ───────────────────────────────────────────────────────────

In [ ]:
# ── CELL 1: Install dependencies ──
import sys

print('Installing base dependencies...')
!pip install -q trimesh pillow numpy matplotlib huggingface_hub

if MODE == 'subject':
    print('Installing subject mode dependencies...')
    !pip install -q rembg[gpu] onnxruntime-gpu
    # InstantMesh
    import os
    if not os.path.exists('InstantMesh'):
        !git clone https://github.com/TencentARC/InstantMesh.git
    %cd InstantMesh
    !pip install -q -r requirements.txt
    %cd ..
elif MODE == 'scene':
    print('Installing scene mode dependencies...')
    !pip install -q torch torchvision timm
    import os
    if not os.path.exists('MiDaS'):
        !git clone https://github.com/isl-org/MiDaS.git

print('✅ Done!')

In [ ]:
# ── CELL 2: Upload your image ──
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import io, os

print('Upload your image:')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
img = Image.open(io.BytesIO(uploaded[fname])).convert('RGB')

# Resize if too large
MAX = 1024
if max(img.size) > MAX:
    img.thumbnail((MAX, MAX), Image.LANCZOS)
    print(f'Resized to {img.size}')

img.save('input.png')
print(f'Loaded: {img.size[0]}x{img.size[1]}')
plt.figure(figsize=(8,5))
plt.imshow(img); plt.axis('off'); plt.title('Input Image'); plt.show()

In [ ]:
# ── CELL 3A: SUBJECT MODE — Background removal ──
if MODE != 'subject':
    print('Skipping (scene mode) — run next cell')
else:
    from rembg import remove
    import numpy as np

    print('Removing background with rembg...')
    with open('input.png', 'rb') as f:
        result = remove(f.read())

    subject = Image.open(io.BytesIO(result)).convert('RGBA')
    subject.save('subject.png')

    # Crop to tight bounding box
    arr = np.array(subject)
    alpha = arr[:,:,3]
    rows = np.any(alpha > 10, axis=1)
    cols = np.any(alpha > 10, axis=0)
    rmin, rmax = np.where(rows)[0][[0,-1]]
    cmin, cmax = np.where(cols)[0][[0,-1]]
    pad = 20
    rmin=max(0,rmin-pad); rmax=min(arr.shape[0],rmax+pad)
    cmin=max(0,cmin-pad); cmax=min(arr.shape[1],cmax+pad)
    cropped = subject.crop((cmin, rmin, cmax, rmax))

    # White background for InstantMesh
    bg = Image.new('RGB', cropped.size, (255,255,255))
    bg.paste(cropped, mask=cropped.split()[3])
    bg.save('subject_clean.png')

    fig, axes = plt.subplots(1,2,figsize=(10,4))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(bg); axes[1].set_title('Subject isolated'); axes[1].axis('off')
    plt.show()
    print('✅ Background removed')

In [ ]:
# ── CELL 3B: SCENE MODE — MiDaS depth estimation ──
if MODE != 'scene':
    print('Skipping (subject mode) — run next cell')
else:
    import torch
    import numpy as np
    import cv2

    print('Loading MiDaS depth model...')
    midas = torch.hub.load('intel-isl/MiDaS', 'DPT_Large')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Using device: {device}')
    midas.to(device).eval()

    transforms = torch.hub.load('intel-isl/MiDaS', 'transforms')
    transform = transforms.dpt_transform

    print('Estimating depth...')
    img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    input_batch = transform(img_cv).to(device)

    with torch.no_grad():
        prediction = midas(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_cv.shape[:2],
            mode='bicubic',
            align_corners=False
        ).squeeze()

    depth = prediction.cpu().numpy()
    depth_norm = (depth - depth.min()) / (depth.max() - depth.min())
    np.save('depth.npy', depth_norm)

    depth_img = Image.fromarray((depth_norm * 255).astype('uint8'))
    depth_img.save('depthmap.png')

    fig, axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
    axes[1].imshow(depth_norm, cmap='plasma'); axes[1].set_title('Depth map (bright = closer)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()
    print('✅ Depth map generated')

In [ ]:
# ── CELL 4A: SUBJECT MODE — InstantMesh 3D reconstruction ──
if MODE != 'subject':
    print('Skipping (scene mode)')
else:
    import subprocess, os
    os.makedirs('outputs', exist_ok=True)

    print('Running InstantMesh (this takes 2-5 mins on T4)...')
    result = subprocess.run([
        'python', 'InstantMesh/run.py',
        'configs/instant-mesh-large.yaml',
        'subject_clean.png',
        '--save_video',
        '--output_path', 'outputs/'
    ], capture_output=True, text=True)

    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])
    else:
        print('✅ 3D model generated!')
        # Find output GLB
        for root, dirs, fls in os.walk('outputs'):
            for f in fls:
                if f.endswith('.glb') or f.endswith('.obj'):
                    print(f'Found: {os.path.join(root, f)}')

In [ ]:
# ── CELL 4B: SCENE MODE — Depthmap → textured 3D mesh ──
if MODE != 'scene':
    print('Skipping (subject mode)')
else:
    import numpy as np
    import struct
    from PIL import Image
    import json, base64, io

    print('Building 3D mesh from depth map...')
    depth = np.load('depth.npy')
    photo = np.array(img.resize((depth.shape[1], depth.shape[0]))) / 255.0

    # Downsample for mesh density control
    SCALE = 4  # increase for denser mesh (slower)
    H, W = depth.shape[0]//SCALE, depth.shape[1]//SCALE
    depth_small = np.array(Image.fromarray(depth).resize((W,H), Image.LANCZOS))
    photo_small = np.array(Image.fromarray((photo*255).astype('uint8')).resize((W,H), Image.LANCZOS)) / 255.0

    HEIGHT_SCALE = 0.5
    positions, colors, uvs, indices = [], [], [], []

    for y in range(H):
        for x in range(W):
            vx = (x/(W-1))*2 - 1
            vy = -((y/(H-1))*2 - 1)
            vz = float(depth_small[y,x]) * HEIGHT_SCALE
            positions.extend([vx, vy, vz])
            r,g,b = photo_small[y,x,:3]
            colors.extend([float(r), float(g), float(b)])
            uvs.extend([x/(W-1), 1-y/(H-1)])

    for y in range(H-1):
        for x in range(W-1):
            a=y*W+x; b=y*W+x+1; c=(y+1)*W+x+1; d=(y+1)*W+x
            indices.extend([a,b,c,a,c,d])

    # Compute normals
    pos_arr = np.array(positions).reshape(-1,3)
    norms = np.zeros_like(pos_arr)
    idx_arr = np.array(indices).reshape(-1,3)
    for tri in idx_arr:
        a,b,c = pos_arr[tri[0]], pos_arr[tri[1]], pos_arr[tri[2]]
        n = np.cross(b-a, c-a)
        l = np.linalg.norm(n)
        if l > 0:
            n /= l
            norms[tri] += n
    lens = np.linalg.norm(norms, axis=1, keepdims=True)
    lens[lens==0] = 1
    norms /= lens

    print(f'Mesh: {len(pos_arr):,} verts, {len(idx_arr):,} tris')

    # Save as OBJ with vertex colors
    col_arr = np.array(colors).reshape(-1,3)
    with open('scene_terrain.obj', 'w') as f:
        f.write('# Scene terrain from photo depth map\n')
        f.write('# Generated by Claude\n\n')
        for i, (v, c) in enumerate(zip(pos_arr, col_arr)):
            f.write(f'v {v[0]:.4f} {v[1]:.4f} {v[2]:.4f} {c[0]:.3f} {c[1]:.3f} {c[2]:.3f}\n')
        f.write('\n')
        for n in norms:
            f.write(f'vn {n[0]:.4f} {n[1]:.4f} {n[2]:.4f}\n')
        f.write('\n')
        for uv in np.array(uvs).reshape(-1,2):
            f.write(f'vt {uv[0]:.4f} {uv[1]:.4f}\n')
        f.write('\n')
        for tri in idx_arr:
            a,b,c = tri+1
            f.write(f'f {a}/{a}/{a} {b}/{b}/{b} {c}/{c}/{c}\n')

    # Also save texture
    img.save('texture.png')
    print('✅ Saved scene_terrain.obj + texture.png')

In [ ]:
# ── CELL 5: Download your files ──
from google.colab import files
import os

if MODE == 'subject':
    for root, dirs, fls in os.walk('outputs'):
        for f in fls:
            if f.endswith(('.glb', '.obj', '.mp4')):
                path = os.path.join(root, f)
                print(f'Downloading {f}...')
                files.download(path)

elif MODE == 'scene':
    print('Downloading scene_terrain.obj...')
    files.download('scene_terrain.obj')
    print('Downloading texture.png...')
    files.download('texture.png')
    print('Downloading depthmap.png...')
    files.download('depthmap.png')

print('\n✅ All done! Load the OBJ/GLB in the 3D viewer HTML file.')